# Land

This notebook is an introduction to analyzing CLM land model results from a global simulation.  It uses results from a case similar to the ones you have been running in the Tutorial, but run for longer in order to provide more output. We've prestaged model results from this simulation and another simulation using a different model configuration in a shared directory. This way you can get started on analyzing simulation results and compare differences caused by model structure. 

## Learning Goals

- Read in global CLM data
- Make basic spatial plots
- Make basic timeseries plots

## Running CUPID's ILAMB from a CLM checkout

CUPiD stands for CESM Unified Postprocessing and Diagnostics. 
ILAMB stands for International Land Model Benchmarking.

### ONE-TIME instructions

Today we have you working on Casper, though the next three lines could (if necessary) be executed on Derecho. Derecho is the preferred option for running CLM and CESM simulations, while Casper is the preferred option for processing model output.

We hope to eventually release a version of the CLM that includes CUPiD. For now we have you clone a branch that tests this idea in a recent version of the CLM.

The following lines:
- clone the branch into a new directory {{ CTSM_CUPID_TAG }}
- enter the directory just created
- update submodules, one of which is CUPiD
- enter the directory tools/CUPiD
- create the conda environment needed to run ILAMB
- check that the cupid-analysis environment got created

```{parsed-literal}
git clone -b branch_tags/{{ '`' + CTSM_CUPID_TAG + '`' }} https://github.com/ESCOMP/ctsm.git {{ '`' + CTSM_CUPID_TAG + '`' }}
cd {{ '`' + CTSM_CUPID_TAG + '`' }}
./bin/git-fleximod update
cd tools/CUPiD

### ILAMB ON CASPER

First we initiate an interactive session that requests specific Casper resources for ILAMB to run smoothly.
Then we set up our environment by loading some modules and activating cupid-analysis. Note that we point to a very specific older version of cupid-analysis.yml here, because newer versions cause ILAMB to fail as far as we know.

```
qsub -I -X -l select=1:ncpus=12:mpiprocs=12:mem=120GB -A <account_num> -q casper@casper-pbs -l walltime=03:00:00
cd examples/external_diag_packages
module load conda
conda activate /glade/campaign/cgd/tss/tutorials/clm_2026/diagnostics/cupid_ilamb/environments/cupid-analysis.yml
module load nco

Before generating ilamb_nohoff_final_CLM_BGC_subset.cfg, you must update config.yml with simulation names, years, etc. pertaining to your cases. Here you could replace BGC_subset with BGC or SP.

```
../../helper_scripts/generate_ilamb_config_files.py --cupid-config-loc . --run-type BGC_subset

The last tool will remind you that you need regridded files to run ILAMB. In this case we will copy LDF’S regridded files that were generated earlier.

```
cp -r /glade/derecho/scratch/<user>/cupid_output/<case>/lnd/proc/regrid /glade/derecho/scratch/<user>/archive/<case>/lnd/hist/.

Note that you should also modify model_setup.txt if necessary at this time. Specifically for timeslice simulations, set “shift” values “N, 2000” where N is the start year of the timeslice simulation's data as set in config.yml. This allows ILAMB to compare timeslice simulations to observed datasets.

Now we set ILAMB_ROOT, and we kick off ilamb-run. **If your shell is tcsh change the next command to "setenv ILAMB_ROOT ../../ilamb_aux"**

```
export ILAMB_ROOT=../../ilamb_aux

mpiexec ilamb-run --config ./ilamb_nohoff_final_CLM_BGC_subset.cfg --build_dir ./ILAMB_output/ --df_errs ./ilamb_aux/quantiles_Whittaker_cmip5v6.parquet --define_regions ./ilamb_aux/DATA/regions/LandRegions.nc ./ilamb_aux/DATA/regions/Whittaker.nc --regions global --model_setup ./model_setup.txt --filter .clm2.h0 >&! ilamb.out &

Plots will appear in ILAMB_output. When ILAMB completes, copy the entire directory to your laptop, initiating the command from your laptop. To view, click index.html.

```
scp -r ILAMB_output <your laptop's desktop>